In [2]:
# 09 LightGBM challenger vs GLM v2 (same target, weight, folds)
# Purpose: does a GBM separate risk better than the balanced ridge GLM?
# Rule fixed before results (DECISIONS.md, Oct 19): adopt GBM or blend only if average validation D2
# over 2023 and 2024 is >= 0.003 higher than GLM v2, with no decile A/E clearly worse.
import sys, time
sys.path.append("..")
import numpy as np
import pandas as pd
import lightgbm as lgb
from src.target import LINES, load_policies, load_claims, build_model_target
from src.features import build_features
from src.evaluate import evaluate
from src.diagnostics import FOLDS
print("lightgbm", lgb.__version__, "| pandas", pd.__version__)

NON_FEATURES = ("row_id", "year", "exposure_clean", "target", "weight")
PARAMS = dict(objective="tweedie", tweedie_variance_power=1.5, metric="tweedie",
              learning_rate=0.05, num_leaves=31, min_data_in_leaf=500,      # large leaves: losses are noisy
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              verbose=-1, seed=42)

# 1. Same modelling table as GLM v2 (excess on every peril); reused if already built in this kernel
if "data" not in globals():
    claims = load_claims()
    data = {}
    for line in LINES:
        p = load_policies(line, "train")
        mt, comp, loading = build_model_target(claims, p, line)
        df = mt.merge(build_features(line, "train", policies=p), on="row_id", how="left", validate="one_to_one")
        data[line] = {"df": df, "loading": loading}
        print(f"{line}: table ready")
glm_vp = pd.read_parquet("../results/glm_v2_valid_preds.parquet")


def prep(X, cats=None):
    """Non-numeric columns -> pandas category with the training categories (unseen levels become missing).
    Works for object, string and pandas 3 'str' dtypes."""
    X = X.copy()
    if cats is None:
        cats = {c: sorted(X[c].dropna().astype(str).unique()) for c in X.columns
                if not pd.api.types.is_numeric_dtype(X[c]) and not pd.api.types.is_bool_dtype(X[c])}
    for c, levels in cats.items():
        X[c] = pd.Categorical(X[c].astype("string"), categories=levels)
    for c in X.columns:
        if pd.api.types.is_bool_dtype(X[c]):
            X[c] = X[c].astype(int)
    return X, cats


def fit_gbm(tr, feats):
    """Number of trees chosen inside the training years: hold out the last training year,
    early-stop there, then refit on all training years with that number of trees."""
    last = tr.year.max()
    inner, hold = tr[tr.year < last], tr[tr.year == last]
    Xi, cats = prep(inner[feats])
    Xh, _ = prep(hold[feats], cats)
    di = lgb.Dataset(Xi, inner.target / inner.weight, weight=inner.weight)
    dh = lgb.Dataset(Xh, hold.target / hold.weight, weight=hold.weight, reference=di)
    b = lgb.train(PARAMS, di, num_boost_round=3000, valid_sets=[dh],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    n_trees = max(int(b.best_iteration * 1.1), 50)            # a little more data -> a few more trees
    Xt, cats = prep(tr[feats])
    model = lgb.train(PARAMS, lgb.Dataset(Xt, tr.target / tr.weight, weight=tr.weight), num_boost_round=n_trees)
    k = tr.target.sum() / (model.predict(Xt) * tr.weight).sum()   # same balance correction as the GLM
    return model, cats, k, n_trees


# 2. Fit on both folds and compare
rows, imp = [], {}
for line in LINES:
    df = data[line]["df"]
    feats = [c for c in df.columns if c not in NON_FEATURES]
    for vy, tys in FOLDS:
        t0 = time.time()
        tr, va = df[df.year.isin(tys)], df[df.year == vy]
        model, cats, k, n_trees = fit_gbm(tr, feats)
        Xv, _ = prep(va[feats], cats)
        p_gbm = k * model.predict(Xv) * va.weight.to_numpy()
        g = glm_vp[(glm_vp.line == line) & (glm_vp.year == vy)].set_index("row_id")
        p_glm = g.loc[va.row_id, "pred"].to_numpy()
        for name, pred in [("GLM", p_glm), ("GBM", p_gbm), ("blend 50/50", 0.5 * p_glm + 0.5 * p_gbm)]:
            r = evaluate(va.target, pred, va.weight, show=False)
            lf = r["lift"]["A/E"]
            rows.append({"line": line, "year": vy, "model": name, "D2": r["D2"], "A/E": r["A/E"],
                         "low": lf.iloc[0], "top": lf.iloc[-1], "trees": n_trees if name == "GBM" else None,
                         "k": k if name == "GBM" else None})
        if vy == 2024:
            imp[line] = pd.Series(model.feature_importance("gain"), index=feats).sort_values(ascending=False)
        print(f"{line} {vy}: {n_trees} trees, k = {k:.3f}, {time.time() - t0:.0f}s")

res = pd.DataFrame(rows)
res.to_csv("../results/lightgbm_vs_glm.csv", index=False)

# 3. Results and the pre-set decision
for line in LINES:
    t = res[res.line == line].pivot(index="model", columns="year", values=["D2", "A/E", "low", "top"])
    t.columns = [f"{m}_{y}" for m, y in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line} =====")
    print(t.sort_values("D2_avg", ascending=False).round(4).to_string())

print("\ndecision rule (gain in average D2 vs GLM v2; threshold 0.003):")
avg = res.groupby(["line", "model"]).D2.mean().unstack()
for line in LINES:
    for m in ["GBM", "blend 50/50"]:
        gain = avg.loc[line, m] - avg.loc[line, "GLM"]
        print(f"  {line:9s} {m:12s} {gain:+.4f}  {'meets threshold' if gain >= 0.003 else 'below threshold'}")

print("\ntop 10 features by gain (2024 fold, share of total gain):")
for line in LINES:
    s = imp[line] / imp[line].sum()
    print(f"  {line}: " + ", ".join(f"{f} {v:.0%}" for f, v in s.head(10).items()))

lightgbm 4.7.0 | pandas 3.0.6
motor 2023: 70 trees, k = 1.095, 5s
motor 2024: 101 trees, k = 1.090, 8s


/var/folders/62/vbb1n0sn0z36rw_dx0zhb7_80000gn/T/ipykernel_85431/549389212.py:43: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = pd.Categorical(X[c].astype("string"), categories=levels)


buildings 2023: 50 trees, k = 1.226, 3s


/var/folders/62/vbb1n0sn0z36rw_dx0zhb7_80000gn/T/ipykernel_85431/549389212.py:43: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X[c] = pd.Categorical(X[c].astype("string"), categories=levels)


buildings 2024: 52 trees, k = 1.174, 4s
contents 2023: 50 trees, k = 1.228, 3s
contents 2024: 50 trees, k = 1.173, 4s

===== motor =====
             D2_2023  D2_2024  A/E_2023  A/E_2024  low_2023  low_2024  top_2023  top_2024  D2_avg
model                                                                                            
blend 50/50   0.0508   0.0524    1.0499    1.0335    0.9128    0.9629    1.1567    1.1274  0.0516
GLM           0.0503   0.0519    1.0266    1.0183    0.9505    1.0087    1.0415    1.0730  0.0511
GBM           0.0461   0.0490    1.0744    1.0492    0.9088    1.0944    1.1925    1.1232  0.0475

===== buildings =====
             D2_2023  D2_2024  A/E_2023  A/E_2024  low_2023  low_2024  top_2023  top_2024  D2_avg
model                                                                                            
blend 50/50   0.0692   0.0714    0.9909    1.0201    0.8506    0.6595    1.1442    1.1881  0.0703
GBM           0.0640   0.0685    0.9901    1.0389    0.8

In [3]:
# 09b Diagnose the GBM, then ONE revision based on the diagnosis (not on validation D2)
# Signs from 09: early stop at the 50-tree floor, k = 1.09-1.23, previous_insurer/occupation top-3 in
# every line -> suspected overfitting of high-cardinality categorical splits.
from scipy.stats import spearmanr

# 1. Cardinality of categorical features
for line in LINES:
    df = data[line]["df"]
    feats = [c for c in df.columns if c not in NON_FEATURES]
    card = {c: df[c].nunique() for c in feats
            if not pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c])}
    print(f"{line}: " + ", ".join(f"{c} {n}" for c, n in sorted(card.items(), key=lambda x: -x[1])))

# 2. Model-free check: is the level ordering stable between 2021-22 and 2023-24?
print("\nstability of one-way cost per weight by level (Spearman rank correlation between periods):")
for line in LINES:
    df = data[line]["df"]
    for f in ["previous_insurer", "occupation", "region"]:
        if f not in df:
            continue
        g = df.assign(period=np.where(df.year <= 2022, "early", "late")).groupby([f, "period"])[["target", "weight"]].sum()
        rate = (g.target / g.weight).unstack()
        share = df.groupby(f).weight.sum() / df.weight.sum()
        rate = rate[share.reindex(rate.index) >= 0.005].dropna()          # levels with >= 0.5% of weight
        rho = spearmanr(rate.early, rate.late).correlation
        spread = rate.mean(axis=1).max() / rate.mean(axis=1).min()
        print(f"  {line:9s} {f:17s} levels {len(rate):3d} | rank corr {rho:+.2f} | max/min rate {spread:.2f}")

# 3. GBM revision: stronger regularisation of categorical splits, slower learning, patient early stopping
PARAMS2 = dict(PARAMS, learning_rate=0.03, min_data_per_group=1000, cat_smooth=50, cat_l2=10,
               max_cat_threshold=8)


def prep2(X, cats=None):
    """As prep, but unseen levels are set to missing explicitly (avoids the pandas warning)."""
    X = X.copy()
    if cats is None:
        cats = {c: sorted(X[c].dropna().astype(str).unique()) for c in X.columns
                if not pd.api.types.is_numeric_dtype(X[c]) and not pd.api.types.is_bool_dtype(X[c])}
    for c, levels in cats.items():
        s = X[c].astype("string")
        X[c] = pd.Categorical(s.where(s.isin(levels)), categories=levels)
    for c in X.columns:
        if pd.api.types.is_bool_dtype(X[c]):
            X[c] = X[c].astype(int)
    return X, cats


def fit_gbm2(tr, feats, params=PARAMS2, patience=300):
    last = tr.year.max()
    inner, hold = tr[tr.year < last], tr[tr.year == last]
    Xi, cats = prep2(inner[feats])
    Xh, _ = prep2(hold[feats], cats)
    di = lgb.Dataset(Xi, inner.target / inner.weight, weight=inner.weight)
    dh = lgb.Dataset(Xh, hold.target / hold.weight, weight=hold.weight, reference=di)
    b = lgb.train(params, di, num_boost_round=5000, valid_sets=[dh],
                  callbacks=[lgb.early_stopping(patience, verbose=False)])
    n_trees = max(int(b.best_iteration * 1.1), 1)
    Xt, cats = prep2(tr[feats])
    model = lgb.train(params, lgb.Dataset(Xt, tr.target / tr.weight, weight=tr.weight), num_boost_round=n_trees)
    k = tr.target.sum() / (model.predict(Xt) * tr.weight).sum()
    return model, cats, k, n_trees


rows2, imp2 = [], {}
for line in LINES:
    df = data[line]["df"]
    feats = [c for c in df.columns if c not in NON_FEATURES]
    for vy, tys in FOLDS:
        t0 = time.time()
        tr, va = df[df.year.isin(tys)], df[df.year == vy]
        model, cats, k, n_trees = fit_gbm2(tr, feats)
        Xv, _ = prep2(va[feats], cats)
        p_gbm = k * model.predict(Xv) * va.weight.to_numpy()
        g = glm_vp[(glm_vp.line == line) & (glm_vp.year == vy)].set_index("row_id")
        p_glm = g.loc[va.row_id, "pred"].to_numpy()
        for name, pred in [("GLM", p_glm), ("GBM2", p_gbm), ("blend 50/50", 0.5 * p_glm + 0.5 * p_gbm)]:
            r = evaluate(va.target, pred, va.weight, show=False)
            lf = r["lift"]["A/E"]
            rows2.append({"line": line, "year": vy, "model": name, "D2": r["D2"], "A/E": r["A/E"],
                          "low": lf.iloc[0], "top": lf.iloc[-1]})
        if vy == 2024:
            imp2[line] = pd.Series(model.feature_importance("gain"), index=feats).sort_values(ascending=False)
        print(f"{line} {vy}: {n_trees} trees, k = {k:.3f}, {time.time() - t0:.0f}s")

res2 = pd.DataFrame(rows2)
res2.to_csv("../results/lightgbm2_vs_glm.csv", index=False)
for line in LINES:
    t = res2[res2.line == line].pivot(index="model", columns="year", values=["D2", "A/E", "low", "top"])
    t.columns = [f"{m}_{y}" for m, y in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line} =====")
    print(t.sort_values("D2_avg", ascending=False).round(4).to_string())

print("\ndecision rule (gain in average D2 vs GLM v2; threshold 0.003):")
avg2 = res2.groupby(["line", "model"]).D2.mean().unstack()
for line in LINES:
    for m in ["GBM2", "blend 50/50"]:
        gain = avg2.loc[line, m] - avg2.loc[line, "GLM"]
        print(f"  {line:9s} {m:12s} {gain:+.4f}  {'meets threshold' if gain >= 0.003 else 'below threshold'}")
print("\ntop 10 features by gain (2024 fold):")
for line in LINES:
    s = imp2[line] / imp2[line].sum()
    print(f"  {line}: " + ", ".join(f"{f} {v:.0%}" for f, v in s.head(10).items()))

motor: previous_insurer 40, occupation 20, region 12, energy_rating 7, body_type 5, credit_band 5, soil_type 5, fuel_type 4, marital_status 4, urban_density 4, property_type 4, cover_type 3, overnight_parking 3, vehicle_use 3, channel 3, transmission 2, payment_method 2, roof_type 2
buildings: previous_insurer 40, occupation 20, region 12, construction_type 11, energy_rating 7, credit_band 5, soil_type 5, marital_status 4, urban_density 4, property_type 4, channel 3, occupancy 2, smoke_alarms 2, payment_method 2, roof_type 2
contents: previous_insurer 40, occupation 20, region 12, energy_rating 7, credit_band 5, soil_type 5, marital_status 4, urban_density 4, property_type 4, security_level 3, channel 3, tenure 2, payment_method 2, roof_type 2

stability of one-way cost per weight by level (Spearman rank correlation between periods):
  motor     previous_insurer  levels  40 | rank corr +0.10 | max/min rate 1.28
  motor     occupation        levels  20 | rank corr +0.51 | max/min rate 2

In [4]:
# 09c Drop features with no stable signal (model-free check in 09b), in BOTH GLM and GBM
# Rule fixed before results: candidates = categorical features with rank corr < 0.3 between 2021-22 and
# 2023-24 in that line; drop if the GLM's average D2 falls by no more than 0.0005 (simpler model preferred).
from src.glm import BandedTweedieGLM

DROP = {"motor": ["previous_insurer"],
        "buildings": ["previous_insurer", "occupation", "region"],
        "contents": ["previous_insurer", "occupation"]}
GLM_CONFIG = {"motor": dict(alpha=0.5, n_bands=10, drop=None, balance=True),
              "buildings": dict(alpha=0.5, n_bands=20, drop=None, balance=True),
              "contents": dict(alpha=0.3, n_bands=20, drop=None, balance=True)}

rows3, imp3, preds3 = [], {}, []
for line in LINES:
    df = data[line]["df"]
    all_feats = [c for c in df.columns if c not in NON_FEATURES]
    variants = {"all": all_feats, "drop all candidates": [c for c in all_feats if c not in DROP[line]]}
    if line == "buildings":   # region judged separately
        variants["drop all but region"] = [c for c in all_feats if c not in ["previous_insurer", "occupation"]]
    for vy, tys in FOLDS:
        tr, va = df[df.year.isin(tys)], df[df.year == vy]
        g = glm_vp[(glm_vp.line == line) & (glm_vp.year == vy)].set_index("row_id")
        out = {"GLM v2 (all)": g.loc[va.row_id, "pred"].to_numpy()}
        for vname, feats in variants.items():
            if vname == "all":
                continue
            t0 = time.time()
            glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
            out[f"GLM {vname}"] = glm.predict_rate(va[feats]) * va.weight.to_numpy()
            model, cats, k, n_trees = fit_gbm2(tr, feats)
            Xv, _ = prep2(va[feats], cats)
            out[f"GBM {vname}"] = k * model.predict(Xv) * va.weight.to_numpy()
            out[f"blend {vname}"] = 0.5 * out[f"GLM {vname}"] + 0.5 * out[f"GBM {vname}"]
            if vy == 2024 and vname == "drop all candidates":
                imp3[line] = pd.Series(model.feature_importance("gain"), index=feats).sort_values(ascending=False)
            print(f"{line} {vy} [{vname}]: GBM {n_trees} trees, k = {k:.3f}, {time.time() - t0:.0f}s")
        for name, pred in out.items():
            r = evaluate(va.target, pred, va.weight, show=False)
            lf = r["lift"]["A/E"]
            rows3.append({"line": line, "year": vy, "model": name, "D2": r["D2"], "A/E": r["A/E"],
                          "low": lf.iloc[0], "top": lf.iloc[-1]})
        preds3.append(pd.DataFrame({"line": line, "year": vy, "row_id": va.row_id.values,
                                    "actual": va.target.values, "weight": va.weight.values,
                                    **{k_: v for k_, v in out.items()}}))

res3 = pd.DataFrame(rows3)
res3.to_csv("../results/feature_drop_glm_gbm.csv", index=False)
pd.concat(preds3, ignore_index=True).to_parquet("../results/feature_drop_valid_preds.parquet", index=False)

for line in LINES:
    t = res3[res3.line == line].pivot(index="model", columns="year", values=["D2", "A/E", "low", "top"])
    t.columns = [f"{m}_{y}" for m, y in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line} =====")
    print(t.sort_values("D2_avg", ascending=False).round(4).to_string())

avg3 = res3.groupby(["line", "model"]).D2.mean().unstack()
print("\nfeature-drop rule (GLM; drop if D2 falls by <= 0.0005):")
for line in LINES:
    for col in [c for c in avg3.columns if c.startswith("GLM drop")]:
        if pd.notna(avg3.loc[line, col]):
            d = avg3.loc[line, col] - avg3.loc[line, "GLM v2 (all)"]
            print(f"  {line:9s} {col:28s} {d:+.4f}  {'drop' if d >= -0.0005 else 'keep features'}")
print("\nGBM / blend vs GLM v2 (threshold 0.003):")
for line in LINES:
    for col in [c for c in avg3.columns if c.startswith(("GBM", "blend"))]:
        if pd.notna(avg3.loc[line, col]):
            d = avg3.loc[line, col] - avg3.loc[line, "GLM v2 (all)"]
            print(f"  {line:9s} {col:28s} {d:+.4f}  {'meets threshold' if d >= 0.003 else 'below'}")
print("\nGBM top features after dropping (2024 fold):")
for line in LINES:
    s = imp3[line] / imp3[line].sum()
    print(f"  {line}: " + ", ".join(f"{f} {v:.0%}" for f, v in s.head(8).items()))

motor 2023 [drop all candidates]: GBM 154 trees, k = 1.089, 28s
motor 2024 [drop all candidates]: GBM 207 trees, k = 1.081, 43s
buildings 2023 [drop all candidates]: GBM 82 trees, k = 1.173, 15s
buildings 2023 [drop all but region]: GBM 68 trees, k = 1.167, 15s
buildings 2024 [drop all candidates]: GBM 125 trees, k = 1.147, 23s
buildings 2024 [drop all but region]: GBM 105 trees, k = 1.142, 23s
contents 2023 [drop all candidates]: GBM 74 trees, k = 1.160, 16s
contents 2024 [drop all candidates]: GBM 125 trees, k = 1.144, 24s

===== motor =====
                           D2_2023  D2_2024  A/E_2023  A/E_2024  low_2023  low_2024  top_2023  top_2024  D2_avg
model                                                                                                          
blend drop all candidates   0.0526   0.0534    1.0536    1.0357    0.9096    0.9996    1.1347    1.1179  0.0530
GLM drop all candidates     0.0504   0.0518    1.0268    1.0182    0.9615    1.0022    1.0338    1.0780  0.0511
GL

In [5]:
# 09d Association between features (inputs only): Spearman (num-num), Cramer's V (cat-cat), eta (num-cat)
# Purpose: find strongly linked features (possible proxies), e.g. what occupation stands in for.
from itertools import combinations
from scipy.stats import chi2_contingency

def is_cat(s):
    return not pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)

def cramers_v(a, b):
    t = pd.crosstab(a, b)
    if t.shape[0] < 2 or t.shape[1] < 2:
        return np.nan
    chi2 = chi2_contingency(t, correction=False)[0]
    return np.sqrt(chi2 / (t.values.sum() * (min(t.shape) - 1)))

def eta(cat, num):
    d = pd.DataFrame({"c": cat, "x": num}).dropna()
    m = d.x.mean()
    between = d.groupby("c").x.agg(["mean", "size"]).pipe(lambda g: (g["size"] * (g["mean"] - m) ** 2).sum())
    total = ((d.x - m) ** 2).sum()
    return np.sqrt(between / total) if total > 0 else np.nan

for line in LINES:
    df = data[line]["df"]
    feats = [c for c in df.columns if c not in NON_FEATURES]
    s = df[feats].sample(min(200_000, len(df)), random_state=0)
    for c in s.columns:
        if pd.api.types.is_bool_dtype(s[c]):
            s[c] = s[c].astype(int)
    pairs = []
    for a, b in combinations(feats, 2):
        ca, cb = is_cat(s[a]), is_cat(s[b])
        if ca and cb:
            v, how = cramers_v(s[a].astype("string").fillna("missing"), s[b].astype("string").fillna("missing")), "Cramer V"
        elif ca or cb:
            c_, n_ = (a, b) if ca else (b, a)
            v, how = eta(s[c_].astype("string").fillna("missing"), s[n_]), "eta"
        else:
            v, how = abs(s[a].corr(s[b], method="spearman")), "|Spearman|"
        pairs.append({"a": a, "b": b, "measure": how, "value": v})
    P = pd.DataFrame(pairs).dropna().sort_values("value", ascending=False)
    P.to_csv(f"../results/feature_association_{line}.csv", index=False)
    print(f"\n===== {line}: 15 most strongly linked pairs =====")
    print(P.head(15).round(3).to_string(index=False))
    occ = P[(P.a == "occupation") | (P.b == "occupation")].head(6)
    print(f"\n{line}: features most linked to occupation")
    print(occ.round(3).to_string(index=False))

/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r


===== motor: 15 most strongly linked pairs =====
                  a                   b    measure  value
      renewal_count     customer_tenure |Spearman|  1.000
         driver_age      years_licensed |Spearman|  0.992
             region  mean_temperature_c        eta  0.941
vehicle_value_group      engine_size_cc |Spearman|  0.936
      freezing_days  mean_temperature_c |Spearman|  0.935
             region       freezing_days        eta  0.915
   dry_summer_index  mean_temperature_c |Spearman|  0.896
             region wind_exposure_index        eta  0.854
      energy_rating        building_age        eta  0.850
             region      wildfire_index        eta  0.831
             region    dry_summer_index        eta  0.827
             region         rainfall_mm        eta  0.792
            channel    discount_applied        eta  0.711
      freezing_days    dry_summer_index |Spearman|  0.705
     wildfire_index  mean_temperature_c |Spearman|  0.692

motor: features most 

/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r


===== buildings: 15 most strongly linked pairs =====
               a                   b    measure  value
   renewal_count     customer_tenure |Spearman|  1.000
          region  mean_temperature_c        eta  0.941
   freezing_days  mean_temperature_c |Spearman|  0.935
          region       freezing_days        eta  0.914
dry_summer_index  mean_temperature_c |Spearman|  0.897
          region wind_exposure_index        eta  0.853
   energy_rating        building_age        eta  0.852
          region      wildfire_index        eta  0.831
          region    dry_summer_index        eta  0.826
          region         rainfall_mm        eta  0.792
     sum_insured            bedrooms |Spearman|  0.757
         channel    discount_applied        eta  0.714
   freezing_days    dry_summer_index |Spearman|  0.705
  wildfire_index  mean_temperature_c |Spearman|  0.692
   freezing_days      wildfire_index |Spearman|  0.682

buildings: features most linked to occupation
               a   

/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
/Users/woojinji/IAA/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r


===== contents: 15 most strongly linked pairs =====
               a                   b    measure  value
   renewal_count     customer_tenure |Spearman|  1.000
          region  mean_temperature_c        eta  0.941
   freezing_days  mean_temperature_c |Spearman|  0.934
          region       freezing_days        eta  0.914
dry_summer_index  mean_temperature_c |Spearman|  0.897
          region wind_exposure_index        eta  0.853
   energy_rating        building_age        eta  0.851
          region      wildfire_index        eta  0.830
          region    dry_summer_index        eta  0.828
          region         rainfall_mm        eta  0.793
         channel    discount_applied        eta  0.711
   freezing_days    dry_summer_index |Spearman|  0.704
  wildfire_index  mean_temperature_c |Spearman|  0.689
   freezing_days      wildfire_index |Spearman|  0.677
   property_type            bedrooms        eta  0.652

contents: features most linked to occupation
               a     

In [6]:
# 09f Re-check two motor assumptions under the corrected target (excess on every peril)
# (1) cost per weight by year should still be level after all adjustments (AI_USE_LOG 17)
# (2) 2023 frequency shift: frequency counts claims with net > 0, which now excludes small liability claims
from src.target import ADJ, match_claims, adjust_claims, completeness

df = data["motor"]["df"]
g = df.groupby("year")[["target", "weight"]].sum()
print("motor cost per weight by year (capped, before loading):")
print((g.target / g.weight).round(1).to_string())
print(f"spread max/min - 1: {(g.target / g.weight).max() / (g.target / g.weight).min() - 1:.1%}  (was about 2%)")

p = load_policies("motor", "train")
claims_ = load_claims()
m = match_claims(claims_, p, "motor", cap=None)                        # excess on every peril (default)
exp_y = p.groupby("year").exposure_clean.sum()
print("\nfrequency per 1,000 exposures (net > 0, completeness-adjusted), and 2023-24 vs 2021-22 ratio:")
for peril in ["own_damage", "third_party_prop", "bodily_injury"]:
    s = m[(m.peril == peril) & (m.net > 0)]
    f = {y: len(s[s.loss_year == y]) / completeness(m, peril, y) / exp_y[y] * 1000 for y in [2021, 2022, 2023, 2024]}
    ratio = ((f[2023] + f[2024]) / 2) / ((f[2021] + f[2022]) / 2)
    print(f"  {peril:17s} " + " ".join(f"{y}: {v:6.2f}" for y, v in f.items()) +
          f" | ratio {ratio:.3f} (assumed {ADJ['motor']['freq_shift'].get(peril, 1.0)})")

motor cost per weight by year (capped, before loading):
year
2021    526.3
2022    523.8
2023    526.7
2024    516.2
spread max/min - 1: 2.0%  (was about 2%)

frequency per 1,000 exposures (net > 0, completeness-adjusted), and 2023-24 vs 2021-22 ratio:
  own_damage        2021:  43.31 2022:  44.16 2023:  41.26 2024:  43.30 | ratio 0.967 (assumed 0.93)
  third_party_prop  2021:  37.40 2022:  38.58 2023:  36.65 2024:  36.11 | ratio 0.958 (assumed 0.95)
  bodily_injury     2021:  11.07 2022:  10.53 2023:  10.44 2024:  10.14 | ratio 0.953 (assumed 1.0)


In [7]:
# 09e Final fit on 2021-2024: GLM v3 (features dropped by the 09c rule) and an experimental blend
# Purpose: (1) glm_v3.csv - candidate submission; (2) blend_v1_experiment.csv - NOT adopted (fails the
# tail condition); submitted only to check that the validation gain carries over to the leaderboard.
from pathlib import Path
from src.target import DATA_DIR

subs_glm, subs_blend, checks = [], [], []
for line in LINES:
    df = data[line]["df"]
    loading = data[line]["loading"]
    feats = [c for c in df.columns if c not in NON_FEATURES and c not in DROP[line]]
    t0 = time.time()
    glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(df[feats], df.target / df.weight, df.weight)

    pt = load_policies(line, "test")
    Ft = build_features(line, "test", policies=pt)
    assert list(Ft.row_id) == list(pt.row_id)
    expo = pt.exposure_clean.to_numpy()
    rate_glm = glm.predict_rate(Ft[feats])
    pred_glm = np.maximum((rate_glm + loading) * expo, 0)
    subs_glm.append(pd.DataFrame({"row_id": Ft.row_id, "prediction": pred_glm}))

    if line == "motor":
        pred_blend, n_trees, k = pred_glm, None, None
    else:
        model, cats, k, n_trees = fit_gbm2(df, feats)     # early stopping on 2024 inside 2021-2024
        Xt, _ = prep2(Ft[feats], cats)
        rate_gbm = k * model.predict(Xt)
        pred_blend = np.maximum((0.5 * rate_glm + 0.5 * rate_gbm + loading) * expo, 0)
    subs_blend.append(pd.DataFrame({"row_id": Ft.row_id, "prediction": pred_blend}))

    # level check against the latest year (2024 predicted level with the same GLM)
    d24 = df[df.year == 2024]
    lvl24 = np.average(glm.predict_rate(d24[feats]), weights=d24.exposure_clean)
    checks.append({"line": line, "glm_k": glm.k_, "gbm_trees": n_trees, "gbm_k": k,
                   "glm test/2024": np.average(rate_glm, weights=expo) / lvl24,
                   "blend/glm total": pred_blend.sum() / pred_glm.sum()})
    print(f"{line}: done in {time.time() - t0:.0f}s")

print(pd.DataFrame(checks).set_index("line").round(3).to_string())

sample = pd.read_parquet(DATA_DIR / "sample_submission.parquet")
out = Path("../submissions"); out.mkdir(exist_ok=True)
for name, parts in [("glm_v3.csv", subs_glm), ("blend_v1_experiment.csv", subs_blend)]:
    sub = pd.concat(parts, ignore_index=True)
    assert len(sub) == len(sample) and set(sub.row_id) == set(sample.row_id) and sub.row_id.is_unique
    assert sub.prediction.notna().all() and (sub.prediction >= 0).all()
    sub = sample[["row_id"]].merge(sub, on="row_id", how="left")
    sub.to_csv(out / name, index=False)
    print(f"wrote {name}: {len(sub):,} rows, total {sub.prediction.sum():,.0f}")

motor: done in 42s
buildings: done in 33s
contents: done in 36s
           glm_k  gbm_trees  gbm_k  glm test/2024  blend/glm total
line                                                              
motor      1.019        NaN    NaN          0.953            1.000
buildings  1.040      168.0   1.14          0.984            0.992
contents   1.051      162.0   1.13          1.010            0.999
wrote glm_v3.csv: 626,639 rows, total 188,239,908
wrote blend_v1_experiment.csv: 626,639 rows, total 187,634,819


In [8]:
# 09f (corrected) Motor frequency by year: three definitions, then the pre-set classification
# A = notebook 03 method (all claims, no completeness)  -> should reproduce the 0.93 / 0.95 shift
# B = all claims, completeness-adjusted               -> the definition used for the adjustment
#     (excess leverage is already handled by the severity trend applied to gross amounts)
# C = net > 0, completeness-adjusted (first 09f)       -> mixes the excess effect into frequency
# Rule fixed before results (on B): both 2023 and 2024 below 2021-22 by a similar amount -> level shift
# (factor = 2023-24 mean / 2021-22 mean); steady change every year -> annual trend; one year only -> none.
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from src.target import ADJ, load_policies, load_claims, match_claims, completeness

YEARS4 = [2021, 2022, 2023, 2024]
p = load_policies("motor", "train")
m = match_claims(load_claims(), p, "motor", cap=None)     # all non-repudiated, de-duplicated claims
exp_y = p.groupby("year").exposure_clean.sum()

rows = []
for peril in sorted(m.peril.unique()):
    comp = {y: completeness(m, peril, y) for y in YEARS4}
    for label, s in [("A all, no completeness", m[m.peril == peril]),
                     ("B all, completeness", m[m.peril == peril]),
                     ("C net>0, completeness", m[(m.peril == peril) & (m.net > 0)])]:
        for y in YEARS4:
            n = (s.loss_year == y).sum()
            c = 1.0 if label.startswith("A") else comp[y]
            rows.append({"peril": peril, "definition": label, "year": y,
                         "freq_per_1000": n / c / exp_y[y] * 1000, "completeness": comp[y]})
F = pd.DataFrame(rows)

t = F.pivot_table(index=["peril", "definition"], columns="year", values="freq_per_1000")
base = (t[2021] + t[2022]) / 2
t["r2023"] = t[2023] / base
t["r2024"] = t[2024] / base
t["r2023_24"] = (t[2023] + t[2024]) / 2 / base
pd.set_option("display.width", 200)
print("frequency per 1,000 exposures and ratios to the 2021-22 average:")
print(t.round(3).to_string())

print("\ncompleteness used in B and C (2024 below 0.8 makes that year's frequency uncertain):")
print(F[F.definition.str.startswith("B")].pivot_table(index="peril", columns="year",
                                                       values="completeness").round(3).to_string())

print("\nshare of claims with zero net cost (falls over time if the fixed excess explains A/B vs C):")
z = m.assign(zero=m.net <= 0).groupby(["peril", "loss_year"]).zero.mean().unstack()
print(z[YEARS4].round(3).to_string())

print("\nclassification on definition B (suggestion; decided tomorrow with motor validation A/E):")
for peril in sorted(m.peril.unique()):
    r = t.loc[(peril, "B all, completeness")]
    yoy = np.diff(np.log([r[y] for y in YEARS4]))
    below23, below24 = r.r2023 < 0.98, r.r2024 < 0.98
    if below23 and below24 and abs(r.r2023 - r.r2024) <= 0.02:
        verdict = f"level shift, factor {r.r2023_24:.3f}"
    elif (yoy > 0).all() or (yoy < 0).all():
        verdict = f"steady trend, {np.expm1(yoy.mean()):+.1%} a year"
    elif below23 != below24:
        verdict = "one year only -> no adjustment"
    else:
        verdict = "no clear change -> no adjustment"
    current = ADJ["motor"]["freq_shift"].get(peril, 1.0)
    flag = "  (2024 completeness < 0.8: uncertain)" if F[(F.peril == peril) & (F.year == 2024)].completeness.iloc[0] < 0.8 else ""
    print(f"  {peril:17s} r2023 {r.r2023:.3f} r2024 {r.r2024:.3f} -> {verdict}  [current shift {current}]{flag}")

frequency per 1,000 exposures and ratios to the 2021-22 average:
year                                       2021    2022    2023    2024  r2023  r2024  r2023_24
peril            definition                                                                    
bodily_injury    A all, no completeness  11.405  10.721  10.261   5.896  0.928  0.533     0.730
                 B all, completeness     11.413  10.796  10.896  10.538  0.981  0.949     0.965
                 C net>0, completeness   11.067  10.529  10.444  10.136  0.967  0.939     0.953
glass_hail       A all, no completeness  19.001  19.725  20.593  20.122  1.064  1.039     1.051
                 B all, completeness     19.001  19.725  20.633  20.734  1.066  1.071     1.068
                 C net>0, completeness   12.352  13.195  14.296  14.555  1.119  1.139     1.129
own_damage       A all, no completeness  63.726  64.452  59.450  57.747  0.928  0.901     0.914
                 B all, completeness     63.726  64.452  59.516  59.834